In [4]:
import torch
import torch.nn as nn
from torchvision import models

In [5]:
class BaselineModel(nn.Module):
    def __init__(self, num_classes = 10, in_channels = 3):
        super(BaselineModel, self).__init__()
        # Layer 1
        self.layer1 = nn.Sequential(
            nn.Conv2d (in_channels, 96, kernel_size = 11, stride = 4, padding = 2),
            nn.ReLU(inplace = True),
            nn.MaxPool2d(kernel_size = 3, stride = 2)
        )

        # Layer 2
        self.layer2 = nn.Sequential(
            nn.Conv2d (96, 256, kernel_size = 5, padding = 2),
            nn.ReLU(inplace = True),
            nn.MaxPool2d(kernel_size = 3, stride = 2)
        )

        # Layer 3
        self.layer3 = nn.Sequential(
            nn.Conv2d (256, 384, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),
        )

        # Layer 4
        self.layer4 = nn.Sequential(
            nn.Conv2d (384, 384, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),
        )

        # Layer 5
        self.layer5 = nn.Sequential(
            nn.Conv2d (384, 256, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),
            nn.MaxPool2d(kernel_size=3,stride = 2)
        )

        # Fully Connected Layer
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )

    # Forward Pass
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        x = self.classifier(x)
        return x

In [6]:
class Improved_Model(nn.Module):
    def __init__(self, num_classes=10):
        super(Improved_Model, self).__init__()
        
        self.features = nn.Sequential(
            # Layer 1
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.BatchNorm2d(64), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Layer 2
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.BatchNorm2d(192), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Layer 3
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384), 
            nn.ReLU(inplace=True),
            
            # Layer 4
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384), 
            nn.ReLU(inplace=True),
            
            # Layer 5
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [7]:
def get_resnet_model(num_classes=10):
    # Load Pre-Trained Model
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    
    # Unfreeze Layer 4     
    for param in model.layer4.parameters():
        param.requires_grad = True
    
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model